# 04 · Feature extraction

| | |
|---|---|
| **input**  | `data/clean/<subject>.csv` |
| **output** | windowed feature matrices + fitted `StandardScaler`s (`.npz` / `.joblib`) |

Pipeline: band-split EEG → chronological split → sliding windows → feature branches → standardisation.

| branch | window | features |
|---|---|---|
| EEG Hjorth | 0.5 s | activity + mobility per channel, per band (α, β) |
| EEG CSP→Hjorth | 0.5 s | CSP (static vs dynamic) then Hjorth on components |
| EMG RMS | 0.2 s | root-mean-square per muscle |
| motion | 0.2 s | mean marker position |
| environment | – | chair height, stair height |

In [ ]:
import sys
from pathlib import Path

# make the `motion_intent` package importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler
from mne.decoding import CSP

from motion_intent.preprocessing import (
    merge_xz_all, drop_single_axis_marker_cols, append_band_columns,
    bandpass_by_session,
)
from motion_intent.windowing import (
    split_by_time_block, make_reference_indices, windows_at, transition_indices,
)
from motion_intent.features import (
    is_dynamic, extract_hjorth_fast, extract_csp_hjorth,
    extract_rms, extract_mean_position, build_env_features,
)

## Load & band-split

In [ ]:
subject = 'subjectA'
df = pd.read_csv(config.CLEAN_DIR / f'{subject}.csv')
df = drop_single_axis_marker_cols(merge_xz_all(df))
df = df.dropna().reset_index(drop=True)

# derive the integer label from the one-hot columns if needed
if 'label' not in df and set(config.CLASS_NAMES).issubset(df.columns):
    df['label'] = df[config.CLASS_NAMES].to_numpy().argmax(axis=1)

# α / β band copies of the sensorimotor EEG channels
df = append_band_columns(df, config.EEG_CH_FEATURE, 'Session', config.FS,
                         bands={k: config.EEG_BANDS[k] for k in ('alpha', 'beta')},
                         drop_original=False)
# broadband copy for the CSP branch
df = bandpass_by_session(df, config.EEG_CH_FEATURE, 'Session', config.FS,
                         band=config.EEG_BROADBAND)
# EMG band-pass
df = bandpass_by_session(df, config.EMG_COLS, 'Session', config.FS,
                         band=config.EMG_BAND)

## Chronological split, per session

In [ ]:
splits = {'train': [], 'val': [], 'test': []}
for _, df_sess in df.groupby('Session'):
    tr, va, te = split_by_time_block(df_sess)
    splits['train'].append(tr); splits['val'].append(va); splits['test'].append(te)

## Windowing

In [ ]:
COLS = {
    'eeg_bb':  config.EEG_CH_FEATURE,
    'eeg_a':   [f'{c}_alpha' for c in config.EEG_CH_FEATURE],
    'eeg_b':   [f'{c}_beta'  for c in config.EEG_CH_FEATURE],
    'emg':     config.EMG_COLS,
    'motion':  config.MOTION_COLS,
}

def window_split(frames):
    out = {k: [] for k in COLS}
    ys = []
    for df_sess in frames:
        t = df_sess['t_sec'].to_numpy()
        ref = make_reference_indices(t, config.WIN_SEC_EEG, config.STEP_SEC)
        y = df_sess['label'].to_numpy()
        ys.append(y[[min(i, len(y) - 1) for i in ref if i >= config.WIN_EEG]])
        for name, cols in COLS.items():
            win = config.WIN_EEG if name.startswith('eeg') else config.WIN_EMG
            out[name].append(windows_at(df_sess[cols].to_numpy(), ref, win))
    return {k: np.concatenate(v) for k, v in out.items()}, np.concatenate(ys)

Xw, yw = {}, {}
for part in ('train', 'val', 'test'):
    Xw[part], yw[part] = window_split(splits[part])

## Fit CSP on the training EEG (static vs dynamic)

In [ ]:
csp = CSP(n_components=config.CSP_N_COMPONENTS, reg='ledoit_wolf',
          log=False, norm_trace=True)
csp.fit(Xw['train']['eeg_bb'], is_dynamic(yw['train']))

## Feature branches

In [ ]:
def features_for(part):
    X = Xw[part]
    hjorth = np.concatenate([extract_hjorth_fast(X['eeg_a']),
                             extract_hjorth_fast(X['eeg_b'])], axis=1)
    csp_hjorth = extract_csp_hjorth(X['eeg_bb'], csp)
    rms = extract_rms(X['emg'])
    motion = extract_mean_position(X['motion'])
    env = build_env_features(len(rms), config.DEFAULT_CHAIR_HEIGHT_M,
                             config.DEFAULT_STAIR_HEIGHT_M)
    return dict(hjorth=hjorth, csp_hjorth=csp_hjorth, rms=rms,
                motion=motion, env=env)

F = {part: features_for(part) for part in ('train', 'val', 'test')}

## Standardise (fit on train) and save

In [ ]:
scalers = {}
for key in ('hjorth', 'csp_hjorth', 'rms', 'motion'):
    sc = StandardScaler().fit(F['train'][key])
    scalers[key] = sc
    for part in ('train', 'val', 'test'):
        F[part][key] = sc.transform(F[part][key])

art_dir = config.DATA_DIR / 'features'
art_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scalers, art_dir / f'{subject}_scalers.joblib')
for part in ('train', 'val', 'test'):
    np.savez(art_dir / f'{subject}_{part}.npz', y=yw[part], **F[part])
print('saved to', art_dir)